# Lab 3: Socket Programming and Protocol Design
**CMSC395 - Computer Networks**  
Estimated time: 4–5 hours | Points: 100

---

Work through each cell in order. Parts 1 and 2 build on each other — complete them in sequence.

> **Submission:** Use **Kernel → Restart Kernel and Run All Cells** before your final push.  
> Commit `lab3_failure.pcap` to your repository.

## Setup - Imports

Run this cell first.

In [ ]:
# Setup — run this cell before starting any part
import asyncio
import socket
import time
import json
import threading
import concurrent.futures
import subprocess
import logging
import re
from pathlib import Path
from collections import defaultdict
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np

LAB_SERVER    = 'lab.millersville.edu'
SERVER_HOST   = '127.0.0.1'
SERVER_PORT   = 9000
FAILURE_PCAP  = Path('lab3_failure.pcap')

# Configure logging so server output is readable in the notebook
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('lab3')

print('Imports OK')
print(f'asyncio version: {asyncio.__version__ if hasattr(asyncio, "__version__") else "stdlib"}')

---
## Part 1: Concurrent Port Scanner with Service Fingerprinting

Build a concurrent port scanner that identifies open ports and fingerprints services
from their banners. Scan only `lab.yourclass.edu` and `localhost`.

### Cell 1.1 - Concurrent Port Scanner

Implement two functions:

- `scan_port(host, port, timeout=2.0)` → `(port, is_open, banner_str)`  
  Connect to host:port. If open, wait up to `timeout` seconds for a banner.  
  If no banner arrives, send `HEAD / HTTP/1.0\r\n\r\n` and wait again.  
  Return the first line of whatever the server sends, or an empty string.

- `scan_range(host, start_port, end_port, max_workers=100)` → `list of (port, banner)`  
  Scan concurrently using `concurrent.futures.ThreadPoolExecutor`.  
  Return only open ports, sorted by port number.

**Hint:** Use `socket.setblocking(False)` with `socket.connect_ex()` and `select.select()`  
for non-blocking connects, or simply use `socket.settimeout(timeout)` with a try/except.

In [ ]:
# Cell 1.1 - Concurrent port scanner
import select

def scan_port(host, port, timeout=2.0):
    """
    Attempt to connect to host:port.
    Returns (port, is_open, banner) where banner is a string (may be empty).
    """
    # Your code here
    pass


def scan_range(host, start_port, end_port, max_workers=100):
    """
    Scan ports start_port..end_port concurrently.
    Returns list of (port, banner) tuples for open ports, sorted by port.
    """
    # Your code here
    pass


# Quick test: scan a small range on localhost to verify the scanner works
print('Testing scanner on localhost ports 8990-9010...')
results = scan_range('127.0.0.1', 8990, 9010, max_workers=20)
if results:
    for port, banner in results:
        print(f'  {port}: {banner[:60]}')
else:
    print('  No open ports found in range (expected if server not yet running)')


### Cell 1.2 - Service Fingerprinting

Implement `identify_service(port, banner)` that returns a service name string.

Required patterns (from the lab guide) plus at least **two additional services** of your choice.  
Document your additional patterns in the markdown cell below.

In [ ]:
# Cell 1.2 — Service fingerprinting

def identify_service(port, banner):
    """
    Identify the service running on a port based on port number and banner.
    Returns a service name string, or 'Unknown' if not identified.
    """
    banner = banner.strip()

    # Required patterns
    if banner.startswith('SSH-'):
        return 'SSH'
    # Your additional patterns here

    # Well-known port fallback
    WELL_KNOWN = {
        21: 'FTP', 22: 'SSH', 23: 'Telnet', 25: 'SMTP',
        53: 'DNS', 80: 'HTTP', 110: 'POP3', 143: 'IMAP',
        443: 'HTTPS', 465: 'SMTPS', 587: 'SMTP-submission',
        993: 'IMAPS', 995: 'POP3S', 3306: 'MySQL',
        5432: 'PostgreSQL', 6379: 'Redis', 8080: 'HTTP-alt',
    }
    return WELL_KNOWN.get(port, 'Unknown')


# Test fingerprinting with sample banners
test_cases = [
    (22,   'SSH-2.0-OpenSSH_8.9p1 Ubuntu-3ubuntu0.1'),
    (25,   '220 mail.example.com ESMTP Postfix'),
    (8080, 'HTTP/1.0 200 OK'),
    (9000, '+PONG'),
    (443,  ''),
]
print(f"{'Port':<6} {'Service':<15} {'Banner':<50}")
print('-' * 72)
for port, banner in test_cases:
    svc = identify_service(port, banner)
    print(f"{port:<6} {svc:<15} {banner[:50]}")


**Additional service patterns:**

*(Document the two additional services you added. What pattern you match and what service it identifies)*

### Cell 1.3 - Speedup Measurement and JSON Report

Scan ports 1–1024 on the lab server:
1. Sequential (max_workers=1)
2. Concurrent (max_workers=100)
3. Plot speedup vs max_workers for values [1, 10, 25, 50, 100, 200]
4. Print the JSON report for the concurrent scan

In [ ]:
# Cell 1.3 - Speedup measurement and JSON report

# Sequential baseline
print('Running sequential scan (max_workers=1)...')
t0 = time.perf_counter()
seq_results = scan_range(LAB_SERVER, 1, 1024, max_workers=1)
seq_time = time.perf_counter() - t0
print(f'Sequential: {seq_time:.2f}s, {len(seq_results)} open ports')

# Concurrent scan
print('\nRunning concurrent scan (max_workers=100)...')
t0 = time.perf_counter()
con_results = scan_range(LAB_SERVER, 1, 1024, max_workers=100)
con_time = time.perf_counter() - t0
speedup = seq_time / con_time
print(f'Concurrent: {con_time:.2f}s, {len(con_results)} open ports, speedup={speedup:.1f}x')

# Speedup vs concurrency plot
worker_counts = [1, 10, 25, 50, 100, 200]
times = []
print('\nMeasuring speedup at different concurrency levels...')
for w in worker_counts:
    t0 = time.perf_counter()
    scan_range(LAB_SERVER, 1, 256, max_workers=w)  # smaller range for speed
    elapsed = time.perf_counter() - t0
    times.append(elapsed)
    print(f'  max_workers={w:<4}: {elapsed:.2f}s')

# Your speedup plot here
fig, ax = plt.subplots(figsize=(9, 4))
# ax.plot(...)
plt.tight_layout()
plt.show()

# Build and print JSON report
report = {
    'target': LAB_SERVER,
    'scan_time_sec': round(con_time, 3),
    'sequential_time_sec': round(seq_time, 3),
    'speedup': round(speedup, 1),
    'open_ports': [
        {'port': port, 'service': identify_service(port, banner), 'banner': banner}
        for port, banner in con_results
    ]
}
print(json.dumps(report, indent=2))


### Reflection 1.A

At what concurrency level does the speedup plateau in your plot, and what limits further gains?
Reference your plot and the specific worker count where diminishing returns begin.

**Your answer:**

*(Write here)*

### Reflection 1.B

Banner grabbing reveals service versions (e.g. `OpenSSH_8.9p1`). From an attacker's perspective,
what is valuable about this information? From a defender's perspective, what can you do about it?

**Your answer:**

*(Write here)*

---
## Part 2: Key-Value Store Server and Client Library

Design a text-based wire protocol, implement a concurrent server, and build a client library.

{: .important }
> **Before writing any code**, complete Cell 2.1.
> Your implementation must be consistent with your documented decisions.

### Cell 2.1 — Protocol Design Decisions

Document your design decisions before writing any implementation code.
Address all four questions from the lab guide. Double-click to edit.

## Protocol Design Decisions

**1. Values containing spaces (e.g. `SET key hello world`):**  
*(Your decision and rationale here)*

**2. Maximum key length:**  
*(Your decision and rationale here)*

**3. Wrong number of arguments:**  
*(What error response does the server return?)*

**4. Forbidden characters in key names:**  
*(Which characters are forbidden and why?)*

### Cell 2.2 - Server Implementation

Implement the key-value store server. The server must:
- Handle multiple concurrent clients with `asyncio` (or threading)
- Support all 7 commands from the protocol specification
- Log every request with timestamp, client address, command, and outcome
- Track uptime, connected client count, and total request count for `STATUS`
- Never crash on bad input. Return `-ERR message\r\n` for every error

After implementing, start the server as a background task.

In [ ]:
# Cell 2.2 - Server implementation

# Shared state
kv_store      = {}         # the key-value store
server_start  = time.time()
client_count  = 0          # currently connected clients
request_count = 0          # total requests served

# Forbidden key characters (adjust to match your Cell 2.1 decisions)
FORBIDDEN_KEY_CHARS = set(' ,+-')
MAX_KEY_LEN = 64


def validate_key(key):
    """
    Return None if key is valid, or an error string if not.
    """
    if not key:
        return 'key cannot be empty'
    if len(key) > MAX_KEY_LEN:
        return f'key too long (max {MAX_KEY_LEN} chars)'
    bad = set(key) & FORBIDDEN_KEY_CHARS
    if bad:
        return f'key contains forbidden characters: {bad}'
    return None


async def handle_command(line):
    """
    Parse and execute one protocol command.
    Returns the response string (including \r\n).
    """
    global request_count
    request_count += 1

    parts = line.strip().split(None, 2)  # split into at most 3 parts
    if not parts:
        return '-ERR empty command\r\n'

    cmd = parts[0].upper()

    # Your command handling here
    if cmd == 'PING':
        return '+PONG\r\n'

    elif cmd == 'STATUS':
        uptime = int(time.time() - server_start)
        return f'+{uptime},{client_count},{request_count}\r\n'

    elif cmd == 'QUIT':
        return '+BYE\r\n'

    elif cmd == 'SET':
        # Your implementation here
        pass

    elif cmd == 'GET':
        # Your implementation here
        pass

    elif cmd == 'DEL':
        # Your implementation here
        pass

    elif cmd == 'KEYS':
        # Your implementation here
        pass

    else:
        return f'-ERR unknown command: {cmd}\r\n'


async def handle_client(reader, writer):
    """
    Handle one connected client until it disconnects or sends QUIT.
    """
    global client_count
    client_count += 1
    addr = writer.get_extra_info('peername')
    log.info(f'Client connected: {addr}  (total: {client_count})')

    try:
        while True:
            try:
                data = await reader.readline()
            except Exception:
                break

            if not data:
                break

            line = data.decode(errors='replace')
            log.info(f'{addr} >> {line.strip()}')

            response = await handle_command(line)
            log.info(f'{addr} << {response.strip()}')

            writer.write(response.encode())
            await writer.drain()

            if response.startswith('+BYE'):
                break

    finally:
        client_count -= 1
        log.info(f'Client disconnected: {addr}  (total: {client_count})')
        try:
            writer.close()
        except Exception:
            pass


async def run_server(host, port):
    """Start the key-value store server."""
    server = await asyncio.start_server(handle_client, host, port)
    addr = server.sockets[0].getsockname()
    log.info(f'KV server listening on {addr[0]}:{addr[1]}')
    async with server:
        await server.serve_forever()


# Start server as a background task
# Cancel any existing server first
try:
    server_task.cancel()
except NameError:
    pass

kv_store.clear()
server_start  = time.time()
client_count  = 0
request_count = 0

server_task = asyncio.create_task(run_server(SERVER_HOST, SERVER_PORT))
await asyncio.sleep(0.2)   # give server time to bind
print(f'Server started on {SERVER_HOST}:{SERVER_PORT}')


### Cell 2.3 — Client Library

Implement the `KVClient` class with all required methods.  
Use Python's `socket` module (not asyncio) for the client — this lets you use it
synchronously from notebook cells without async/await boilerplate.

Required methods: `connect`, `disconnect`, `set`, `get`, `delete`, `keys`,
`status`, `ping`, `quit`, `repl`, `run_script`.

In [ ]:
# Cell 2.3 — KVClient class

class KVClient:
    """
    Client library for the CMSC395 key-value store protocol.
    """

    def __init__(self, host, port):
        self.host = host
        self.port = port
        self._sock = None
        self._file = None   # socket wrapped as file for readline

    def connect(self):
        """Open a TCP connection to the server."""
        self._sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self._sock.connect((self.host, self.port))
        self._file = self._sock.makefile('r', encoding='utf-8')

    def disconnect(self):
        """Close the connection."""
        if self._sock:
            try:
                self._sock.close()
            except Exception:
                pass
            self._sock = None
            self._file = None

    def _send(self, command):
        """
        Send a command string (will add \r\n if not present).
        Returns the server response line (stripped).
        """
        if not command.endswith('\r\n'):
            command += '\r\n'
        self._sock.sendall(command.encode())
        return self._file.readline().strip()

    def ping(self):
        """Send PING. Returns True if server responds +PONG."""
        resp = self._send('PING')
        return resp == '+PONG'

    def set(self, key, value):
        """
        Set key to value. Returns True on success.
        Raises ValueError on error.
        """
        # Your code here
        pass

    def get(self, key):
        """
        Get value for key. Returns the value string.
        Raises KeyError if key not found.
        """
        # Your code here
        pass

    def delete(self, key):
        """
        Delete key. Returns True on success.
        Raises KeyError if key not found.
        """
        # Your code here
        pass

    def keys(self):
        """
        Return list of all keys. Returns empty list if store is empty.
        """
        # Your code here
        pass

    def status(self):
        """
        Return server status as dict: {uptime, clients, requests}.
        """
        # Your code here
        pass

    def quit(self):
        """Send QUIT and close the connection."""
        try:
            self._send('QUIT')
        finally:
            self.disconnect()

    def repl(self):
        """
        Interactive REPL: read commands from stdin, print responses.
        Exit with QUIT or Ctrl+C. Best used from a terminal, not a notebook.
        """
        print(f'Connected to {self.host}:{self.port}. Type QUIT to exit.')
        try:
            while True:
                try:
                    cmd = input('> ').strip()
                except EOFError:
                    break
                if not cmd:
                    continue
                resp = self._send(cmd)
                print(resp)
                if cmd.upper() == 'QUIT':
                    break
        except KeyboardInterrupt:
            print('\nInterrupted')

    def run_script(self, filename):
        """
        Execute commands from a file (one per line).
        Print each command and its response.
        """
        with open(filename) as f:
            for line in f:
                cmd = line.strip()
                if not cmd or cmd.startswith('#'):
                    continue
                print(f'>> {cmd}')
                resp = self._send(cmd)
                print(f'<< {resp}')


# Quick connection test
client = KVClient(SERVER_HOST, SERVER_PORT)
client.connect()
print(f'PING: {client.ping()}')
print(f'STATUS: {client.status()}')
client.disconnect()
print('Connection test passed')


### Cell 2.4 — Functional Test

Exercise every command and error condition. Required scenarios:
- SET and GET a value
- GET a non-existent key
- SET a key with a value containing spaces
- DEL a key and confirm it is gone
- KEYS before and after adding multiple keys
- STATUS twice to show request counter incrementing
- An invalid command — show the error response

In [ ]:
# Cell 2.4 — Functional test

client = KVClient(SERVER_HOST, SERVER_PORT)
client.connect()

def test(description, fn):
    """Run a test case and print the result."""
    print(f'\n--- {description} ---')
    try:
        result = fn()
        print(f'Result: {result}')
    except Exception as e:
        print(f'Exception ({type(e).__name__}): {e}')


# SET and GET
test('SET name=alice', lambda: client.set('name', 'alice'))
test('GET name', lambda: client.get('name'))

# GET non-existent key
test('GET missing key (expect KeyError)', lambda: client.get('does_not_exist'))

# Value with spaces — show your protocol's handling
test('SET greeting = "hello world"', lambda: client.set('greeting', 'hello world'))
test('GET greeting', lambda: client.get('greeting'))

# DEL
test('DEL name', lambda: client.delete('name'))
test('GET name after DEL (expect KeyError)', lambda: client.get('name'))

# KEYS
test('KEYS (empty)', lambda: client.keys())
client.set('a', '1')
client.set('b', '2')
client.set('c', '3')
test('KEYS after 3 SETs', lambda: client.keys())

# STATUS twice
test('STATUS (first)', lambda: client.status())
test('STATUS (second — request count should increase)', lambda: client.status())

# Invalid command
print('\n--- Invalid command (expect -ERR) ---')
resp = client._send('INVALID arg1 arg2')
print(f'Response: {resp}')

client.disconnect()
print('\nFunctional test complete')


### Cell 2.5 — Batch Mode Test

Create a script file with at least 10 commands (including comments and blank lines).
Run it with `client.run_script()` and leave the output visible.

In [ ]:
# Cell 2.5 — Batch mode test

# Write the script file
script_content = """# CMSC395 Lab 3 — batch test script
# This script exercises the key-value store protocol

PING
STATUS
SET course CMSC395
SET semester Fall2026
SET instructor TBD
KEYS
GET course
GET semester
DEL instructor
KEYS
STATUS
QUIT
"""

script_path = Path('lab3_batch.txt')
script_path.write_text(script_content)
print(f'Script written to {script_path}')
print('=' * 50)

# Run the script
client = KVClient(SERVER_HOST, SERVER_PORT)
client.connect()
client.run_script(script_path)
# Note: run_script handles QUIT which closes the connection


### Reflection 2.A

Your server uses a shared key-value store across all clients. If you used `asyncio`, explain
why concurrent access to the store is safe without locks. If you used threading, explain
what you locked and why.

**Your answer:**

*(Write here)*

### Reflection 2.B

You chose a specific approach for values containing spaces. What are the tradeoffs of
your approach compared to alternatives such as length-prefixed values, quoted values,
or base64 encoding?

**Your answer:**

*(Write here)*

### Reflection 2.C

The `STATUS` command exposes internal server state to any connected client. In a production
system, what are the security implications of this? How would you restrict access?

**Your answer:**

*(Write here)*

---
## Part 3: Failure Handling and Observability

Extend your server and client with heartbeat detection, reconnection logic,
and a live failure demonstration.

### Cell 3.1 - Heartbeat Implementation

Extend the server and client:

**Server:** after each command, check when the last `PING` was received from this client.
If more than 30 seconds have passed without a `PING`, close the connection and log the event.
Use `asyncio.wait_for()` to implement the timeout:

```python
try:
    data = await asyncio.wait_for(reader.readline(), timeout=30.0)
except asyncio.TimeoutError:
    log.warning(f'{addr} heartbeat timeout — closing connection')
    break
```

**Client:** add `start_heartbeat(interval=10)` that sends `PING` every `interval` seconds
in a background thread, and `stop_heartbeat()` that stops it.

Demonstrate that a client which stops sending heartbeats is disconnected by the server
within ~30 seconds.

In [ ]:
# Cell 3.1 - Heartbeat implementation
# Extend handle_client() to use asyncio.wait_for with a 30s timeout
# Extend KVClient with start_heartbeat() and stop_heartbeat()

# --- Updated server with heartbeat timeout ---

HEARTBEAT_TIMEOUT = 30.0   # seconds

async def handle_client_hb(reader, writer):
    """
    Updated client handler with heartbeat timeout.
    Extends handle_client() from Cell 2.2.
    """
    global client_count
    client_count += 1
    addr = writer.get_extra_info('peername')
    log.info(f'Client connected: {addr}')

    try:
        while True:
            try:
                # Your heartbeat timeout implementation here
                data = await asyncio.wait_for(
                    reader.readline(),
                    timeout=HEARTBEAT_TIMEOUT
                )
            except asyncio.TimeoutError:
                log.warning(f'{addr} heartbeat timeout — closing')
                break
            except Exception:
                break

            if not data:
                break

            line = data.decode(errors='replace')
            log.info(f'{addr} >> {line.strip()}')
            response = await handle_command(line)
            log.info(f'{addr} << {response.strip()}')
            writer.write(response.encode())
            await writer.drain()

            if response.startswith('+BYE'):
                break
    finally:
        client_count -= 1
        try:
            writer.close()
        except Exception:
            pass


# --- Restart server with heartbeat handler ---
try:
    server_task.cancel()
    await asyncio.sleep(0.1)
except Exception:
    pass

kv_store.clear()
server_start  = time.time()
client_count  = 0
request_count = 0

async def run_server_hb(host, port):
    server = await asyncio.start_server(handle_client_hb, host, port)
    log.info(f'KV server (heartbeat) on {host}:{port}')
    async with server:
        await server.serve_forever()

server_task = asyncio.create_task(run_server_hb(SERVER_HOST, SERVER_PORT))
await asyncio.sleep(0.2)
print('Heartbeat server started')


# --- KVClient with heartbeat ---

class KVClientHB(KVClient):
    """
    KVClient extended with heartbeat support.
    """

    def __init__(self, host, port):
        super().__init__(host, port)
        self._hb_thread = None
        self._hb_stop   = threading.Event()

    def start_heartbeat(self, interval=10):
        """
        Send PING every interval seconds in a background thread.
        """
        self._hb_stop.clear()

        def _hb_loop():
            while not self._hb_stop.wait(timeout=interval):
                try:
                    ok = self.ping()
                    log.info(f'Heartbeat: {"OK" if ok else "FAILED"}')
                except Exception as e:
                    log.warning(f'Heartbeat error: {e}')
                    break

        self._hb_thread = threading.Thread(target=_hb_loop, daemon=True)
        self._hb_thread.start()
        log.info(f'Heartbeat started (interval={interval}s)')

    def stop_heartbeat(self):
        """Stop the background heartbeat thread."""
        self._hb_stop.set()
        if self._hb_thread:
            self._hb_thread.join(timeout=2)
        log.info('Heartbeat stopped')


# Demonstrate: connect, start heartbeat, then stop it and wait for server timeout
# Note: 30s timeout means you'll need to wait ~30s to see the server disconnect
# For demonstration, reduce HEARTBEAT_TIMEOUT to 5 seconds above
print('\nDemonstration: connect with heartbeat, then stop heartbeat')
print('(Server will disconnect client after timeout)')
client_hb = KVClientHB(SERVER_HOST, SERVER_PORT)
client_hb.connect()
client_hb.start_heartbeat(interval=2)   # fast heartbeat for demo
await asyncio.sleep(6)                   # let a few heartbeats fire
print('Stopping heartbeat...')
client_hb.stop_heartbeat()
print('Heartbeat stopped — server will disconnect after timeout')
# Wait for server to detect the missing heartbeat
await asyncio.sleep(HEARTBEAT_TIMEOUT + 2)
print('Demonstration complete — check server log above for disconnect message')


### Cell 3.2 - Reconnection with Exponential Backoff

Add reconnection logic to `KVClientHB.connect()`. Demonstrate by starting
the client before the server is running, show the retry output,
then start the server and confirm the client connects.

In [ ]:
# Cell 3.2 — Reconnection with exponential backoff

class KVClientReconnect(KVClientHB):
    """
    KVClientHB extended with exponential backoff reconnection.
    """

    def connect(self, max_retries=5):
        """
        Connect with exponential backoff retry.
        Raises ConnectionError if all retries fail.
        """
        delay = 1
        for attempt in range(max_retries):
            try:
                super().connect()
                if attempt > 0:
                    log.info(f'Connected after {attempt + 1} attempts')
                return
            except (ConnectionRefusedError, OSError) as e:
                if attempt == max_retries - 1:
                    raise ConnectionError(
                        f'Failed to connect after {max_retries} attempts'
                    ) from e
                log.warning(f'Connection failed (attempt {attempt + 1}/{max_retries}), '
                            f'retrying in {delay}s... ({e})')
                time.sleep(delay)
                delay *= 2


# Demonstrate: stop the server, try to connect, then restart server
print('Stopping server to demonstrate reconnection...')
server_task.cancel()
await asyncio.sleep(0.2)
print('Server stopped')

print('\nAttempting connection (will fail and retry)...')
reconnect_client = KVClientReconnect(SERVER_HOST, SERVER_PORT)

# Start reconnection in a thread so we can restart the server concurrently
connect_result = []

def try_connect():
    try:
        reconnect_client.connect(max_retries=4)
        connect_result.append('success')
    except ConnectionError as e:
        connect_result.append(f'failed: {e}')

connect_thread = threading.Thread(target=try_connect, daemon=True)
connect_thread.start()

# Restart the server after 3 seconds
await asyncio.sleep(3)
print('\nRestarting server...')
kv_store.clear()
server_start  = time.time()
client_count  = 0
request_count = 0
server_task = asyncio.create_task(run_server_hb(SERVER_HOST, SERVER_PORT))
await asyncio.sleep(0.2)
print('Server restarted')

connect_thread.join(timeout=10)
print(f'\nConnection result: {connect_result[0] if connect_result else "timeout"}')

if connect_result and connect_result[0] == 'success':
    print(f'PING after reconnect: {reconnect_client.ping()}')
    reconnect_client.disconnect()


### Cell 3.3 - Live Failure Demonstration

Before running this cell, start a `tshark` capture in a terminal:

```bash
tshark -i lo -w ~/lab3_failure.pcap -f "tcp port 9000" -a duration:60
```

Then run this cell. After it completes, copy the capture:

```bash
cp ~/lab3_failure.pcap ./lab3_failure.pcap
git add lab3_failure.pcap
git commit -m "Lab3: add failure demonstration capture"
```

In [ ]:
# Cell 3.3 - Live failure demonstration
# Start tshark in a terminal BEFORE running this cell.

print('=== Live Failure Demonstration ===')
print()

# Step 1: Connect client with heartbeat
print('[1] Connecting client with heartbeat...')
demo_client = KVClientReconnect(SERVER_HOST, SERVER_PORT)
demo_client.connect()
demo_client.start_heartbeat(interval=5)

# Step 2: Store some data
demo_client.set('demo_key', 'alive')
print(f'[2] Stored key: demo_key = {demo_client.get("demo_key")}')
await asyncio.sleep(3)

# Step 3: Kill the server
print('[3] Killing server...')
server_task.cancel()
try:
    await server_task
except asyncio.CancelledError:
    pass
print('    Server stopped')

# Step 4: Wait — heartbeat will detect the failure
print('[4] Waiting for heartbeat to detect failure...')
await asyncio.sleep(8)

# Step 5: Restart server
print('[5] Restarting server...')
kv_store.clear()
server_start  = time.time()
client_count  = 0
request_count = 0
server_task = asyncio.create_task(run_server_hb(SERVER_HOST, SERVER_PORT))
await asyncio.sleep(0.5)
print('    Server restarted')

# Step 6: Client reconnects automatically
print('[6] Reconnecting client...')
demo_client.stop_heartbeat()
demo_client.disconnect()

demo_client2 = KVClientReconnect(SERVER_HOST, SERVER_PORT)
demo_client2.connect(max_retries=3)
demo_client2.start_heartbeat(interval=5)
print(f'    Reconnected. PING: {demo_client2.ping()}')
print(f'    STATUS: {demo_client2.status()}')

demo_client2.stop_heartbeat()
demo_client2.disconnect()

print()
print('=== Demonstration complete ===')
print('Copy lab3_failure.pcap from your home directory to this repo directory.')


### Reflection 3.A

Look at `lab3_failure.pcap` in Wireshark. Did the connection close with a TCP FIN or a TCP RST
when the server was killed? What does each mean, and what caused the one you observed?

**Your answer:**

*(Write here. Reference specific packets from your capture)*

### Reflection 3.B

Your exponential backoff reaches a maximum of 16 seconds before giving up. In a production
system, what factors would determine the right maximum retry delay and retry count?
What is the risk of retrying too aggressively?

**Your answer:**

*(Write here)*

### Reflection 3.C

The heartbeat mechanism detects a failed client and closes the connection.
What resources does the server reclaim when it closes an idle connection?
What would happen over time if it did not?

**Your answer:**

*(Write here)*

---
## Submission Checklist

Before your final push, complete this checklist. Replace each `[ ]` with `[x]` when done.

- [ ] Cell 1.1: Scanner works, test against localhost shown
- [ ] Cell 1.2: Fingerprinting identifies required services plus 2 additional
- [ ] Cell 1.3: Speedup plot shown, JSON report printed
- [ ] Reflections 1.A and 1.B completed
- [ ] Cell 2.1: Protocol design decisions documented for all 4 questions
- [ ] Cell 2.2: Server starts, log output visible
- [ ] Cell 2.3: KVClient connection test passes
- [ ] Cell 2.4: Functional test covers all required scenarios
- [ ] Cell 2.5: Batch mode test with 10+ commands shown
- [ ] Reflections 2.A, 2.B, and 2.C completed
- [ ] Cell 3.1: Heartbeat demo shows server disconnecting idle client
- [ ] Cell 3.2: Reconnection demo shows retry output and successful reconnect
- [ ] Cell 3.3: Full failure demonstration output visible
- [ ] `lab3_failure.pcap` committed to repository
- [ ] Reflections 3.A, 3.B, and 3.C completed
- [ ] Notebook runs cleanly with Kernel → Restart Kernel and Run All Cells
- [ ] At least 5 commits with descriptive messages in repository history

Final commit message must be exactly: `Lab3 final submission`

---
*CMSC395 Computer Networks - Lab 3*  
*Push this notebook to your repository with the message: `Lab3 final submission`*